In [5]:

import os

dataset_path = r"C:\Users\HP\OneDrive\Desktop\HeavySuppliersWarehouseDatasets"

print("My current path:", dataset_path)
print("Folder exists:", os.path.isdir(dataset_path))




My current path: C:\Users\HP\OneDrive\Desktop\HeavySuppliersWarehouseDatasets
Folder exists: True


In [11]:

dataset_path = os.path.join(
    dataset_path,
    "HeavySuppliersWarehouseDatasets"
)

print("Folder exists:", os.path.isdir(dataset_path))
print("Files found:", len(os.listdir(dataset_path)))

for file in os.listdir(dataset_path):
    print(file)



Folder exists: True
Files found: 12
branches.csv
customers.csv
inventory_master.csv
invoices.csv
payments.csv
products.csv
purchase_orders_header.csv
purchase_orders_lines.csv
sales_orders_header.csv
sales_orders_lines.csv
stock_ledger.csv
suppliers.csv


In [8]:

import os

print("Files and folders inside dataset folder:\n")

for item in os.listdir(dataset_path):
    print(item)


Files and folders inside dataset folder:

HeavySuppliersWarehouseDatasets


In [12]:

# Load all 12 CSV datasets

datasets = {}

for file in os.listdir(dataset_path):
    if file.lower().endswith(".csv"):
        name = file[:-4]
        file_path = os.path.join(dataset_path, file)

        datasets[name] = pd.read_csv(file_path)

print("All datasets loaded successfully!")
print("Total datasets:", len(datasets))

# Display dataset dimensions
print("\nDATASET SUMMARY")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows and {df.shape[1]} columns")


All datasets loaded successfully!
Total datasets: 12

DATASET SUMMARY
branches: 6 rows and 13 columns
customers: 500 rows and 15 columns
inventory_master: 180 rows and 8 columns
invoices: 18033 rows and 10 columns
payments: 19257 rows and 5 columns
products: 30 rows and 23 columns
purchase_orders_header: 24000 rows and 10 columns
purchase_orders_lines: 155495 rows and 9 columns
sales_orders_header: 20000 rows and 11 columns
sales_orders_lines: 130402 rows and 9 columns
stock_ledger: 237230 rows and 9 columns
suppliers: 8 rows and 12 columns


In [13]:

# Display all column names from every dataset

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("=" * 60)

    print("Total Columns:", len(df.columns))

    for column in df.columns:
        print("-", column)



DATASET: branches
Total Columns: 13
- branch_id
- branch_name
- city
- state
- region
- warehouse_type
- warehouse_capacity
- service_center_available
- manager_id
- total_employees
- avg_monthly_revenue
- monthly_operational_cost
- market_demand_index

DATASET: customers
Total Columns: 15
- customer_id
- customer_type
- industry_segment
- city
- state
- pincode
- region
- branch_id
- credit_limit
- current_balance
- payment_terms
- customer_since
- last_purchase_date
- total_purchase_value
- customer_rating

DATASET: inventory_master
Total Columns: 8
- product_id
- branch_id
- opening_stock
- reorder_level
- safety_stock
- max_stock
- current_stock
- warehouse_bin

DATASET: invoices
Total Columns: 10
- invoice_id
- so_id
- customer_id
- branch_id
- invoice_date
- due_date
- total_order_value
- total_gst_amount
- grand_total
- payment_status

DATASET: payments
Total Columns: 5
- payment_id
- invoice_id
- payment_date
- payment_amount
- payment_method

DATASET: products
Total Columns: 

In [14]:

# Display column names and their data types

for name, df in datasets.items():

    print("\nDATASET:", name)
    print("-" * 50)

    print(df.dtypes)



DATASET: branches
--------------------------------------------------
branch_id                   object
branch_name                 object
city                        object
state                       object
region                      object
warehouse_type              object
warehouse_capacity          object
service_center_available    object
manager_id                   int64
total_employees              int64
avg_monthly_revenue          int64
monthly_operational_cost     int64
market_demand_index          int64
dtype: object

DATASET: customers
--------------------------------------------------
customer_id             object
customer_type           object
industry_segment        object
city                    object
state                   object
pincode                  int64
region                  object
branch_id               object
credit_limit             int64
current_balance          int64
payment_terms           object
customer_since          object
last_purchase_date

In [15]:

# Candidate primary keys for our datasets

primary_keys = {
    "branches": ["branch_id"],
    "customers": ["customer_id"],
    "inventory_master": ["product_id", "branch_id"],
    "invoices": ["invoice_id"],
    "payments": ["payment_id"],
    "products": ["product_id"],
    "purchase_orders_header": ["po_id"],
    "purchase_orders_lines": ["po_id", "line_number"],
    "sales_orders_header": ["so_id"],
    "sales_orders_lines": ["so_id", "line_number"],
    "suppliers": ["supplier_id"],
    "stock_ledger": ["ledger_id"]
}

# Validate primary keys

for name, keys in primary_keys.items():

    if name not in datasets:
        print(name, "- Dataset not found")
        continue

    df = datasets[name]

    if not all(key in df.columns for key in keys):
        print(name, "- Check key columns:", keys)
        continue

    missing = df[keys].isnull().any(axis=1).sum()
    duplicates = df.duplicated(subset=keys).sum()

    print("\nDataset:", name)
    print("Candidate key:", keys)
    print("Missing key records:", missing)
    print("Duplicate key records:", duplicates)



Dataset: branches
Candidate key: ['branch_id']
Missing key records: 0
Duplicate key records: 0

Dataset: customers
Candidate key: ['customer_id']
Missing key records: 0
Duplicate key records: 0

Dataset: inventory_master
Candidate key: ['product_id', 'branch_id']
Missing key records: 0
Duplicate key records: 0

Dataset: invoices
Candidate key: ['invoice_id']
Missing key records: 0
Duplicate key records: 197

Dataset: payments
Candidate key: ['payment_id']
Missing key records: 0
Duplicate key records: 202

Dataset: products
Candidate key: ['product_id']
Missing key records: 0
Duplicate key records: 0

Dataset: purchase_orders_header
Candidate key: ['po_id']
Missing key records: 0
Duplicate key records: 0

Dataset: purchase_orders_lines
Candidate key: ['po_id', 'line_number']
Missing key records: 0
Duplicate key records: 0

Dataset: sales_orders_header
Candidate key: ['so_id']
Missing key records: 0
Duplicate key records: 0

Dataset: sales_orders_lines
Candidate key: ['so_id', 'line_num

In [16]:

# Candidate foreign key relationships

relationships = [
    ("customers", "customer_id", "sales_orders_header", "customer_id"),
    ("products", "product_id", "inventory_master", "product_id"),
    ("products", "product_id", "sales_orders_lines", "product_id"),
    ("suppliers", "supplier_id", "purchase_orders_header", "supplier_id"),
    ("branches", "branch_id", "inventory_master", "branch_id"),
    ("branches", "branch_id", "sales_orders_header", "branch_id"),
    ("purchase_orders_header", "po_id", "purchase_orders_lines", "po_id"),
    ("sales_orders_header", "so_id", "sales_orders_lines", "so_id"),
    ("invoices", "invoice_id", "payments", "invoice_id")
]

for parent, parent_key, child, child_key in relationships:

    parent_df = datasets[parent]
    child_df = datasets[child]

    parent_ids = parent_df[parent_key].dropna().unique()

    unmatched = (
        child_df[child_key].notna()
        & ~child_df[child_key].isin(parent_ids)
    ).sum()

    print(
        f"{parent} -> {child}: "
        f"{unmatched} unmatched records"
    )


customers -> sales_orders_header: 0 unmatched records
products -> inventory_master: 0 unmatched records
products -> sales_orders_lines: 0 unmatched records
suppliers -> purchase_orders_header: 0 unmatched records
branches -> inventory_master: 0 unmatched records
branches -> sales_orders_header: 0 unmatched records
purchase_orders_header -> purchase_orders_lines: 0 unmatched records
sales_orders_header -> sales_orders_lines: 0 unmatched records
invoices -> payments: 0 unmatched records


In [17]:

# Check missing values in all datasets

missing_summary = []

for name, df in datasets.items():

    total_missing = df.isnull().sum().sum()

    missing_summary.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Total_Columns": len(df.columns),
        "Missing_Values": total_missing
    })

missing_df = pd.DataFrame(missing_summary)

print(missing_df.to_string(index=False))


               Dataset  Total_Rows  Total_Columns  Missing_Values
              branches           6             13               0
             customers         500             15               0
      inventory_master         180              8               0
              invoices       18033             10               0
              payments       19257              5               0
              products          30             23               0
purchase_orders_header       24000             10            2370
 purchase_orders_lines      155495              9               0
   sales_orders_header       20000             11               0
    sales_orders_lines      130402              9               0
          stock_ledger      237230              9               0
             suppliers           8             12               0


In [18]:

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) > 0:
        print("\nDataset:", name)
        print(missing)



Dataset: purchase_orders_header
received_date    2370
dtype: int64


In [19]:

# Investigate missing received dates

po_df = datasets["purchase_orders_header"]

missing_received = po_df[
    po_df["received_date"].isnull()
]

print("Missing received dates:",
      len(missing_received))

print("\nPurchase order status:")
print(missing_received["po_status"].value_counts(dropna=False))

print("\nOverall purchase order status:")
print(po_df["po_status"].value_counts(dropna=False))


Missing received dates: 2370

Purchase order status:
po_status
Cancelled    2370
Name: count, dtype: int64

Overall purchase order status:
po_status
Received     21630
Cancelled     2370
Name: count, dtype: int64


In [20]:

# Check complete duplicate rows in all datasets

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    print(
        f"{name}: {duplicate_count} duplicate rows"
    )


branches: 0 duplicate rows
customers: 0 duplicate rows
inventory_master: 0 duplicate rows
invoices: 0 duplicate rows
payments: 0 duplicate rows
products: 0 duplicate rows
purchase_orders_header: 0 duplicate rows
purchase_orders_lines: 0 duplicate rows
sales_orders_header: 0 duplicate rows
sales_orders_lines: 0 duplicate rows
stock_ledger: 0 duplicate rows
suppliers: 0 duplicate rows


In [21]:

# Identify columns that may contain dates

for name, df in datasets.items():

    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    if date_columns:
        print("\nDataset:", name)

        for col in date_columns:
            print(
                col,
                "->",
                df[col].dtype
            )



Dataset: customers
last_purchase_date -> object

Dataset: invoices
invoice_date -> object
due_date -> object

Dataset: payments
payment_date -> object

Dataset: products
last_purchase_date -> object

Dataset: purchase_orders_header
order_date -> object
expected_delivery_date -> object
received_date -> object

Dataset: sales_orders_header
order_date -> object
delivery_date -> object

Dataset: stock_ledger
movement_date -> object


In [22]:

# Create cleaned copies of all datasets

cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

# Date columns to convert

date_columns = {
    "customers": ["last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "purchase_orders_header": [
        "order_date",
        "expected_delivery_date",
        "received_date"
    ],
    "sales_orders_header": [
        "order_date",
        "delivery_date"
    ],
    "stock_ledger": ["movement_date"]
}

# Convert date columns

for name, columns in date_columns.items():

    for col in columns:

        cleaned_datasets[name][col] = pd.to_datetime(
            cleaned_datasets[name][col],
            errors="coerce"
        )

print("Date conversion completed!")


Date conversion completed!


In [23]:

for name, columns in date_columns.items():

    print("\nDataset:", name)

    for col in columns:
        print(
            col,
            "->",
            cleaned_datasets[name][col].dtype
        )



Dataset: customers
last_purchase_date -> datetime64[ns]

Dataset: invoices
invoice_date -> datetime64[ns]
due_date -> datetime64[ns]

Dataset: payments
payment_date -> datetime64[ns]

Dataset: products
last_purchase_date -> datetime64[ns]

Dataset: purchase_orders_header
order_date -> datetime64[ns]
expected_delivery_date -> datetime64[ns]
received_date -> datetime64[ns]

Dataset: sales_orders_header
order_date -> datetime64[ns]
delivery_date -> datetime64[ns]

Dataset: stock_ledger
movement_date -> datetime64[ns]


In [24]:

# Check important numeric columns

numeric_checks = {
    "products": ["unit_cost", "unit_price"],
    "sales_orders_lines": ["quantity", "unit_price"],
    "purchase_orders_lines": ["quantity", "unit_cost"],
    "inventory_master": ["opening_stock", "current_stock",
                         "reorder_level", "safety_stock"],
    "branches": ["warehouse_capacity"]
}

for name, columns in numeric_checks.items():

    df = cleaned_datasets[name]

    print("\nDataset:", name)

    for col in columns:

        negative = (df[col] < 0).sum()
        zero = (df[col] == 0).sum()

        print(
            f"{col}: Negative = {negative}, Zero = {zero}"
        )



Dataset: products
unit_cost: Negative = 0, Zero = 0
unit_price: Negative = 0, Zero = 0

Dataset: sales_orders_lines
quantity: Negative = 0, Zero = 0
unit_price: Negative = 0, Zero = 0

Dataset: purchase_orders_lines
quantity: Negative = 0, Zero = 0
unit_cost: Negative = 0, Zero = 0

Dataset: inventory_master
opening_stock: Negative = 0, Zero = 0
current_stock: Negative = 0, Zero = 0
reorder_level: Negative = 0, Zero = 0
safety_stock: Negative = 0, Zero = 0

Dataset: branches


TypeError: '<' not supported between instances of 'str' and 'int'

In [25]:
TypeError: '<' not supported between instances of 'str' and 'int'

SyntaxError: invalid syntax (1928453869.py, line 1)

In [26]:

# Inspect warehouse capacity values

branches_df = cleaned_datasets["branches"]

print("Data type:")
print(branches_df["warehouse_capacity"].dtype)

print("\nWarehouse capacity values:")
print(branches_df["warehouse_capacity"].to_list())

print("\nUnique values:")
print(branches_df["warehouse_capacity"].unique())


Data type:
object

Warehouse capacity values:
['45230 sqft', '38750 sqft', '41890 sqft', '36210 sqft', '29880 sqft', '34120 sqft']

Unique values:
['45230 sqft' '38750 sqft' '41890 sqft' '36210 sqft' '29880 sqft'
 '34120 sqft']


In [27]:

print(cleaned_datasets["branches"].dtypes)


branch_id                   object
branch_name                 object
city                        object
state                       object
region                      object
warehouse_type              object
warehouse_capacity          object
service_center_available    object
manager_id                   int64
total_employees              int64
avg_monthly_revenue          int64
monthly_operational_cost     int64
market_demand_index          int64
dtype: object


In [28]:

branches_df = cleaned_datasets["branches"]

print("Warehouse capacity values:")
print(branches_df["warehouse_capacity"].to_list())

print("\nUnique values:")
print(branches_df["warehouse_capacity"].unique())


Warehouse capacity values:
['45230 sqft', '38750 sqft', '41890 sqft', '36210 sqft', '29880 sqft', '34120 sqft']

Unique values:
['45230 sqft' '38750 sqft' '41890 sqft' '36210 sqft' '29880 sqft'
 '34120 sqft']


In [29]:
# Create a new numeric column
cleaned_datasets["branches"]["warehouse_capacity_sqft"] = (
    cleaned_datasets["branches"]["warehouse_capacity"]
    .str.replace(" sqft", "", regex=False)
    .astype(float)
)

# Check the result
print(cleaned_datasets["branches"][
    ["warehouse_capacity", "warehouse_capacity_sqft"]
])

# Check data type
print(cleaned_datasets["branches"]["warehouse_capacity_sqft"].dtype)

  warehouse_capacity  warehouse_capacity_sqft
0         45230 sqft                  45230.0
1         38750 sqft                  38750.0
2         41890 sqft                  41890.0
3         36210 sqft                  36210.0
4         29880 sqft                  29880.0
5         34120 sqft                  34120.0
float64


In [30]:
capacity = cleaned_datasets["branches"]["warehouse_capacity_sqft"]

print("Negative values:", (capacity < 0).sum())
print("Zero values:", (capacity == 0).sum())
print("Missing values:", capacity.isna().sum())
print("Data type:", capacity.dtype)

Negative values: 0
Zero values: 0
Missing values: 0
Data type: float64


In [31]:
for name, df in cleaned_datasets.items():
    print(f"\n--- {name} ---")
    print(df.dtypes[df.dtypes == "object"])


--- branches ---
branch_id                   object
branch_name                 object
city                        object
state                       object
region                      object
warehouse_type              object
warehouse_capacity          object
service_center_available    object
dtype: object

--- customers ---
customer_id         object
customer_type       object
industry_segment    object
city                object
state               object
region              object
branch_id           object
payment_terms       object
customer_since      object
dtype: object

--- inventory_master ---
product_id       object
branch_id        object
warehouse_bin    object
dtype: object

--- invoices ---
invoice_id        object
so_id             object
customer_id       object
branch_id         object
payment_status    object
dtype: object

--- payments ---
payment_id        object
invoice_id        object
payment_method    object
dtype: object

--- products ---
product_id        

In [32]:
print("CUSTOMER SINCE")
print(cleaned_datasets["customers"]["customer_since"].head(10))
print("\nUnique customer_since formats:")
print(cleaned_datasets["customers"]["customer_since"].unique()[:10])

print("\nPRODUCT DIMENSIONS")
print(cleaned_datasets["products"]["dimensions_cm"].head(10))
print("\nUnique dimensions:")
print(cleaned_datasets["products"]["dimensions_cm"].unique()[:10])


CUSTOMER SINCE
0    2024-04-10
1    2022-01-19
2    2017-02-14
3    2017-12-20
4    2023-07-27
5    2021-10-14
6    2019-03-17
7    2015-06-24
8    2022-04-07
9    2015-12-30
Name: customer_since, dtype: object

Unique customer_since formats:
['2024-04-10' '2022-01-19' '2017-02-14' '2017-12-20' '2023-07-27'
 '2021-10-14' '2019-03-17' '2015-06-24' '2022-04-07' '2015-12-30']

PRODUCT DIMENSIONS
0     40x25x25
1       10x8x8
2     18x12x12
3    180x40x35
4     20x12x12
5     30x20x20
6       90x4x4
7       12x6x4
8     32x18x18
9     40x25x25
Name: dimensions_cm, dtype: object

Unique dimensions:
['40x25x25' '10x8x8' '18x12x12' '180x40x35' '20x12x12' '30x20x20' '90x4x4'
 '12x6x4' '32x18x18' '120x18x18']


In [33]:
cleaned_datasets["customers"]["customer_since"] = pd.to_datetime(
    cleaned_datasets["customers"]["customer_since"],
    errors="coerce"
)

print(cleaned_datasets["customers"]["customer_since"].dtype)
print("Missing dates:",
      cleaned_datasets["customers"]["customer_since"].isna().sum())

datetime64[ns]
Missing dates: 0


In [34]:
# Split dimensions into three separate columns
dimensions = cleaned_datasets["products"]["dimensions_cm"].str.split(
    "x", expand=True
)

# Create numeric columns
cleaned_datasets["products"]["length_cm"] = pd.to_numeric(
    dimensions[0], errors="coerce"
)

cleaned_datasets["products"]["width_cm"] = pd.to_numeric(
    dimensions[1], errors="coerce"
)

cleaned_datasets["products"]["height_cm"] = pd.to_numeric(
    dimensions[2], errors="coerce"
)

# Display results
print(cleaned_datasets["products"][
    ["dimensions_cm", "length_cm", "width_cm", "height_cm"]
].head(10))

  dimensions_cm  length_cm  width_cm  height_cm
0      40x25x25         40        25         25
1        10x8x8         10         8          8
2      18x12x12         18        12         12
3     180x40x35        180        40         35
4      20x12x12         20        12         12
5      30x20x20         30        20         20
6        90x4x4         90         4          4
7        12x6x4         12         6          4
8      32x18x18         32        18         18
9      40x25x25         40        25         25


In [35]:
cleaned_datasets["products"]["volume_cm3"] = (
    cleaned_datasets["products"]["length_cm"]
    * cleaned_datasets["products"]["width_cm"]
    * cleaned_datasets["products"]["height_cm"]
)

print(cleaned_datasets["products"][
    ["product_name", "length_cm", "width_cm",
     "height_cm", "volume_cm3"]
].head(10))

              product_name  length_cm  width_cm  height_cm  volume_cm3
0    Hydraulic Pump HP-300         40        25         25       25000
1  Engine Oil Filter OF-90         10         8          8         640
2        Air Filter AF-120         18        12         12        2592
3        Track Chain TC-45        180        40         35      252000
4     Fuel Injector FI-220         20        12         12        2880
5       Alternator ALT-500         30        20         20       12000
6     Hydraulic Hose HH-60         90         4          4        1440
7       Bucket Tooth BT-10         12         6          4         288
8     Starter Motor SM-700         32        18         18       10368
9      Turbocharger TB-900         40        25         25       25000


In [36]:
products_df = cleaned_datasets["products"]

dimension_columns = [
    "length_cm",
    "width_cm",
    "height_cm"
]

for col in dimension_columns:
    print(f"\n{col}")
    print("Missing:", products_df[col].isna().sum())
    print("Zero:", (products_df[col] == 0).sum())
    print("Negative:", (products_df[col] < 0).sum())

print("\nVolume validation")
print("Missing volume:", products_df["volume_cm3"].isna().sum())
print("Zero volume:", (products_df["volume_cm3"] == 0).sum())
print("Negative volume:", (products_df["volume_cm3"] < 0).sum())



length_cm
Missing: 0
Zero: 0
Negative: 0

width_cm
Missing: 0
Zero: 0
Negative: 0

height_cm
Missing: 0
Zero: 0
Negative: 0

Volume validation
Missing volume: 0
Zero volume: 0
Negative volume: 0


In [37]:
for name, df in cleaned_datasets.items():
    numeric_cols = df.select_dtypes(include="number").columns

    print(f"\n--- {name} ---")

    for col in numeric_cols:
        negative = (df[col] < 0).sum()

        if negative > 0:
            print(f"{col}: {negative} negative values")
            


--- branches ---

--- customers ---

--- inventory_master ---

--- invoices ---

--- payments ---

--- products ---

--- purchase_orders_header ---

--- purchase_orders_lines ---

--- sales_orders_header ---

--- sales_orders_lines ---

--- stock_ledger ---

--- suppliers ---


In [38]:
for name, df in cleaned_datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n--- {name} ---")
        print(missing)
    else:
        print(f"\n--- {name} ---")
        print("No missing values")


--- branches ---
No missing values

--- customers ---
No missing values

--- inventory_master ---
No missing values

--- invoices ---
No missing values

--- payments ---
No missing values

--- products ---
No missing values

--- purchase_orders_header ---
received_date    2370
dtype: int64

--- purchase_orders_lines ---
No missing values

--- sales_orders_header ---
No missing values

--- sales_orders_lines ---
No missing values

--- stock_ledger ---
No missing values

--- suppliers ---
No missing values


In [39]:
po_df = cleaned_datasets["purchase_orders_header"]

print(pd.crosstab(
    po_df["po_status"],
    po_df["received_date"].isna(),
    margins=True
))

received_date  False  True    All
po_status                        
Cancelled          0  2370   2370
Received       21630     0  21630
All            21630  2370  24000


In [40]:
primary_keys = {
    "branches": "branch_id",
    "customers": "customer_id",
    "inventory_master": None,
    "invoices": "invoice_id",
    "payments": "payment_id",
    "products": "product_id",
    "purchase_orders_header": "po_id",
    "sales_orders_header": "so_id",
    "stock_ledger": "movement_id",
    "suppliers": "supplier_id"
}

for name, key in primary_keys.items():
    if key is not None:
        df = cleaned_datasets[name]

        print(
            f"{name} | Duplicate {key}:",
            df[key].duplicated().sum()
        )

branches | Duplicate branch_id: 0
customers | Duplicate customer_id: 0
invoices | Duplicate invoice_id: 197
payments | Duplicate payment_id: 202
products | Duplicate product_id: 0
purchase_orders_header | Duplicate po_id: 0
sales_orders_header | Duplicate so_id: 0
stock_ledger | Duplicate movement_id: 0
suppliers | Duplicate supplier_id: 0


In [41]:
# Find duplicate invoice records
invoices_df = cleaned_datasets["invoices"]

duplicate_invoices = invoices_df[
    invoices_df["invoice_id"].duplicated(keep=False)
].sort_values("invoice_id")

print("Total duplicate invoice rows:",
      len(duplicate_invoices))

print("\nSample duplicate invoice records:")
print(duplicate_invoices.head(20))

Total duplicate invoice rows: 393

Sample duplicate invoice records:
       invoice_id      so_id customer_id branch_id invoice_date   due_date  \
14917  INV-105711  SO-901766       C0208    KOL001   2021-11-20 2022-01-04   
8324   INV-105711  SO-549003       C0191    KOL001   2024-12-15 2025-01-14   
14643  INV-106596  SO-807417       C0281    KOL001   2024-01-25 2024-03-25   
9894   INV-106596  SO-399413       C0409    PUN001   2021-06-16 2021-07-31   
1712   INV-111054  SO-233671       C0072    PUN001   2023-04-11 2023-04-26   
1706   INV-111054  SO-316263       C0313    AHM001   2019-08-02 2019-10-01   
13853  INV-112235  SO-236576       C0220    KOL001   2021-04-17 2021-06-16   
946    INV-112235  SO-363636       C0198    AHM001   2021-01-08 2021-03-09   
13492  INV-112378  SO-911241       C0082    AHM001   2025-01-01 2025-03-02   
1609   INV-112378  SO-870768       C0104    KOL001   2019-04-15 2019-06-14   
14375  INV-124878  SO-170742       C0426    KOL001   2022-05-30 2022-07-1

In [42]:
payments_df = cleaned_datasets["payments"]

duplicate_payments = payments_df[
    payments_df["payment_id"].duplicated(keep=False)
].sort_values("payment_id")

print("Total duplicate payment rows:",
      len(duplicate_payments))

print("\nSample duplicate payment records:")
print(duplicate_payments.head(20))

Total duplicate payment rows: 403

Sample duplicate payment records:
       payment_id  invoice_id payment_date  payment_amount payment_method
19220  PAY-102690  INV-826669   2021-04-23      3837051.40    Credit Card
7121   PAY-102690  INV-196859   2021-12-29       520518.80           Cash
16981  PAY-105592  INV-963449   2023-02-23      2123180.75           Cash
11803  PAY-105592  INV-323176   2022-07-10      1019902.14         Cheque
15115  PAY-112476  INV-743633   2024-06-17      4544341.00    Credit Card
5761   PAY-112476  INV-374653   2024-05-16      1724913.40           Cash
1148   PAY-115474  INV-645256   2022-12-23       473838.80    Credit Card
3929   PAY-115474  INV-583030   2023-08-25      3764707.00    Credit Card
5117   PAY-115921  INV-301855   2022-01-06       353798.70            UPI
9009   PAY-115921  INV-121000   2021-05-08       246614.08           Cash
3460   PAY-117803  INV-450834   2024-05-31       519996.42    Credit Card
9851   PAY-117803  INV-613514   2019-08-22 

In [43]:
for name, key in [
    ("invoices", "invoice_id"),
    ("payments", "payment_id")
]:
    df = cleaned_datasets[name]

    duplicate_ids = df[
        df[key].duplicated(keep=False)
    ]

    print(f"\n--- {name.upper()} ---")
    print("Total records:", len(df))
    print("Unique IDs:", df[key].nunique())
    print("Repeated ID values:",
          df.loc[df[key].duplicated(keep=False), key].nunique())
    print("Extra duplicate occurrences:",
          df[key].duplicated().sum())
    print("Rows affected:", len(duplicate_ids))


--- INVOICES ---
Total records: 18033
Unique IDs: 17836
Repeated ID values: 196
Extra duplicate occurrences: 197
Rows affected: 393

--- PAYMENTS ---
Total records: 19257
Unique IDs: 19055
Repeated ID values: 201
Extra duplicate occurrences: 202
Rows affected: 403
